# 06 Spark Structured Streaming Kafka to Parquet

## Purpose
Read Open-Meteo air-quality events with Spark Structured Streaming, validate the explicit Phase-5 event contract, enrich valid events with static city context, and persist Parquet outputs.

The preferred FH evidence path is Kafka. If Kafka or the Spark Kafka connector is unavailable and fallback is explicitly allowed, the notebook uses the same Spark Structured Streaming transformations with a local JSONL file source. The fallback proves local mechanics only; it is not evidence that Spark read Kafka.

## Inputs

- Kafka topic configured through `.env`, or local Phase-5 JSONL events as an explicit fallback
- `data/silver/city_reference.parquet`
- `data/silver/city_metadata.parquet`

## Outputs

- `data/bronze/open_meteo_stream/`
- `data/bronze/open_meteo_stream_rejects/`
- `data/silver/open_meteo_city_hourly/`
- `data/gold/live_air_quality_latest.parquet/`
- `data/checkpoints/open_meteo_stream_*/`

## Technologies used
PySpark, Spark Structured Streaming, Kafka source connector, explicit `StructType`, static-stream joins, Parquet, checkpointing.

The implementation follows the course reference notebooks for Spark DataFrames, Kafka streams, JSON parsing, grouping, and Parquet read-back.

## Configuration

Use `SPARK_KAFKA_MODE=kafka` and `ALLOW_SPARK_KAFKA_MOCK_FALLBACK=false` for the strict FH JupyterHub evidence run.

Use `SPARK_KAFKA_MODE=auto` locally. The notebook first checks broker reachability and the Spark Kafka connector. If either is unavailable, it transparently selects the local file-stream mock when `ALLOW_SPARK_KAFKA_MOCK_FALLBACK=true`.

`SPARK_MASTER_URL` is read from `.env`. Local Parquet output is reliable with `local[*]`. A remote Spark master is supported only when `DATA_DIR` and `CHECKPOINT_DIR` refer to storage visible to the workers.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import shutil
import socket

_cwd = Path.cwd().resolve()
_candidate_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
load_dotenv(_candidate_root / ".env")

_env_root = os.getenv("PROJECT_ROOT")
PROJECT_ROOT = Path(_env_root).resolve() if _env_root else _candidate_root

def project_path(env_name: str, default: str) -> Path:
    path = Path(os.getenv(env_name, default))
    return path if path.is_absolute() else PROJECT_ROOT / path

DATA_DIR = project_path("DATA_DIR", "data")
CHECKPOINT_DIR = project_path("CHECKPOINT_DIR", "data/checkpoints")
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
CITY_METADATA_PATH = DATA_DIR / "silver" / "city_metadata.parquet"
PHASE5_EVENTS_PATH = DATA_DIR / "bronze" / "open_meteo_raw" / "open_meteo_air_quality_events.jsonl"
PHASE5_SAMPLE_EVENTS_PATH = DATA_DIR / "samples" / "open_meteo_phase5_events_sample.jsonl"

BRONZE_STREAM_PATH = DATA_DIR / "bronze" / "open_meteo_stream"
REJECTS_STREAM_PATH = DATA_DIR / "bronze" / "open_meteo_stream_rejects"
SILVER_STREAM_PATH = DATA_DIR / "silver" / "open_meteo_city_hourly"
LATEST_SNAPSHOT_PATH = DATA_DIR / "gold" / "live_air_quality_latest.parquet"
MOCK_INPUT_DIR = DATA_DIR / "bronze" / "open_meteo_stream_mock_input"

CHECKPOINT_BRONZE = CHECKPOINT_DIR / "open_meteo_stream_bronze"
CHECKPOINT_REJECTS = CHECKPOINT_DIR / "open_meteo_stream_rejects"
CHECKPOINT_SILVER = CHECKPOINT_DIR / "open_meteo_stream_silver"

SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
SPARK_KAFKA_MODE = os.getenv("SPARK_KAFKA_MODE", os.getenv("KAFKA_MODE", "auto")).lower()
ALLOW_SPARK_KAFKA_MOCK_FALLBACK = os.getenv("ALLOW_SPARK_KAFKA_MOCK_FALLBACK", "true").lower() == "true"
SPARK_KAFKA_CONNECTOR_PACKAGE = os.getenv("SPARK_KAFKA_CONNECTOR_PACKAGE", "").strip()

assert SPARK_KAFKA_MODE in {"auto", "kafka", "mock"}, (
    f"SPARK_KAFKA_MODE must be auto, kafka, or mock; got {SPARK_KAFKA_MODE!r}"
)
for required_path in [CITY_REFERENCE_PATH, CITY_METADATA_PATH]:
    assert required_path.exists(), f"Missing Phase dependency: {required_path}"
if SPARK_MASTER_URL.startswith("spark://"):
    print("WARNING: Remote Spark master selected. Use only worker-visible DATA_DIR and CHECKPOINT_DIR paths.")

print({
    "project_root": str(PROJECT_ROOT),
    "spark_master": SPARK_MASTER_URL,
    "spark_kafka_mode": SPARK_KAFKA_MODE,
    "kafka_bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka_topic": KAFKA_TOPIC,
    "data_dir": str(DATA_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
})

## Implementation

### Start Spark and select the streaming source

The Kafka path uses `readStream.format("kafka")`. The local fallback deliberately remains a Spark streaming path: it copies the validated Phase-5 JSONL batch into a local input directory and uses `readStream.text()`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, desc, from_json, lit, row_number, to_timestamp, when
)
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
from pyspark.sql.window import Window

builder = SparkSession.builder.appName("phase6-spark-streaming-kafka-to-parquet").master(SPARK_MASTER_URL)
if SPARK_KAFKA_CONNECTOR_PACKAGE:
    builder = builder.config("spark.jars.packages", SPARK_KAFKA_CONNECTOR_PACKAGE)
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel(os.getenv("LOG_LEVEL", "WARN"))

event_schema = StructType([
    StructField("event_id", StringType(), nullable=True),
    StructField("schema_version", StringType(), nullable=True),
    StructField("source", StringType(), nullable=True),
    StructField("city_id", StringType(), nullable=True),
    StructField("event_time_utc", StringType(), nullable=True),
    StructField("ingestion_time_utc", StringType(), nullable=True),
    StructField("data_status", StringType(), nullable=True),
    StructField("pm2_5", DoubleType(), nullable=True),
    StructField("pm10", DoubleType(), nullable=True),
    StructField("no2", DoubleType(), nullable=True),
])

def kafka_configured() -> bool:
    return "<" not in KAFKA_BOOTSTRAP_SERVERS and "gXX" not in KAFKA_TOPIC

def broker_reachable(bootstrap_servers: str, timeout_seconds: float = 2.0) -> tuple[bool, str | None]:
    try:
        host, port = bootstrap_servers.rsplit(":", 1)
        with socket.create_connection((host, int(port)), timeout=timeout_seconds):
            return True, None
    except Exception as exc:
        return False, str(exc)

def kafka_raw_stream():
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .load()
        .select(
            col("topic").alias("kafka_topic"),
            col("partition"),
            col("offset"),
            col("timestamp").alias("kafka_timestamp"),
            col("key").cast("string").alias("kafka_key"),
            col("value").cast("string").alias("raw_json"),
        )
    )

def mock_raw_stream():
    source_path = PHASE5_EVENTS_PATH if PHASE5_EVENTS_PATH.exists() else PHASE5_SAMPLE_EVENTS_PATH
    assert source_path.exists(), (
        "Missing local Phase-5 JSONL events. Run notebook 05 before notebook 06."
    )
    for path in [
        MOCK_INPUT_DIR,
        BRONZE_STREAM_PATH,
        REJECTS_STREAM_PATH,
        SILVER_STREAM_PATH,
        CHECKPOINT_BRONZE,
        CHECKPOINT_REJECTS,
        CHECKPOINT_SILVER,
    ]:
        shutil.rmtree(path, ignore_errors=True)
    MOCK_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source_path, MOCK_INPUT_DIR / "phase5_events.jsonl")
    return (
        spark.readStream.schema("value string").text(str(MOCK_INPUT_DIR))
        .select(
            lit(f"mock:{KAFKA_TOPIC}").alias("kafka_topic"),
            lit(-1).cast("int").alias("partition"),
            lit(-1).cast("long").alias("offset"),
            current_timestamp().alias("kafka_timestamp"),
            lit(None).cast("string").alias("kafka_key"),
            col("value").alias("raw_json"),
        )
    )

source_mode = "mock"
fallback_reason = None
if SPARK_KAFKA_MODE in {"auto", "kafka"}:
    if not kafka_configured():
        fallback_reason = "Kafka broker or group-specific topic still contains a placeholder."
    else:
        reachable, broker_error = broker_reachable(KAFKA_BOOTSTRAP_SERVERS)
        if not reachable:
            fallback_reason = f"Kafka broker TCP check failed: {broker_error}"
        else:
            try:
                raw_stream = kafka_raw_stream()
                source_mode = "kafka"
            except Exception as exc:
                fallback_reason = f"Spark Kafka source initialization failed: {exc}"

if source_mode != "kafka":
    if SPARK_KAFKA_MODE == "kafka" or not ALLOW_SPARK_KAFKA_MOCK_FALLBACK:
        raise RuntimeError(f"Strict Kafka mode failed: {fallback_reason}")
    raw_stream = mock_raw_stream()

print({
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "selected_source_mode": source_mode,
    "fallback_reason": fallback_reason,
})
event_schema

### Parse, validate, and enrich events

Kafka metadata is retained. JSON is parsed with an explicit schema. Invalid JSON, invalid fields, and unknown city IDs become visible reject rows instead of being silently discarded.

In [ ]:
parsed_stream = (
    raw_stream
    .withColumn("event", from_json(col("raw_json"), event_schema))
)

json_valid_stream = (
    parsed_stream
    .filter(col("event").isNotNull())
    .select(
        "kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json",
        col("event.*"),
    )
)

required_fields_valid = (
    col("event_id").isNotNull()
    & col("schema_version").isNotNull()
    & col("source").isNotNull()
    & col("city_id").isNotNull()
    & col("event_time_utc").isNotNull()
    & col("ingestion_time_utc").isNotNull()
)
schema_valid = (col("schema_version") == lit("1.0")) & (col("source") == lit("open_meteo"))
pollutants_plausible = (
    (col("pm2_5").isNull() | col("pm2_5").between(0, 1000))
    & (col("pm10").isNull() | col("pm10").between(0, 2000))
    & (col("no2").isNull() | col("no2").between(0, 1000))
)

quality_marked_stream = (
    json_valid_stream
    .withColumn("event_time_ts", to_timestamp("event_time_utc"))
    .withColumn("ingestion_time_ts", to_timestamp("ingestion_time_utc"))
    .withColumn("is_required_fields_valid", required_fields_valid)
    .withColumn("is_schema_valid", schema_valid)
    .withColumn("is_pollutant_range_plausible", pollutants_plausible)
    .withColumn(
        "is_event_valid",
        col("is_required_fields_valid")
        & col("is_schema_valid")
        & col("is_pollutant_range_plausible")
        & col("event_time_ts").isNotNull()
        & col("ingestion_time_ts").isNotNull(),
    )
)

quality_valid_stream = quality_marked_stream.filter(col("is_event_valid")).dropDuplicates(["event_id"])
quality_rejects_stream = (
    quality_marked_stream.filter(~col("is_event_valid"))
    .withColumn("reject_reason", lit("event_quality_validation_failed"))
)
json_rejects_stream = (
    parsed_stream.filter(col("event").isNull())
    .select("kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json")
    .withColumn("reject_reason", lit("invalid_json"))
)

city_reference_df = spark.read.parquet(str(CITY_REFERENCE_PATH))
city_metadata_df = spark.read.parquet(str(CITY_METADATA_PATH))
required_city_ref_cols = {"city_id", "city_name", "country_code", "latitude", "longitude"}
required_city_meta_cols = {"city_id", "population", "area_km2", "population_density", "parse_status"}
assert not (required_city_ref_cols - set(city_reference_df.columns)), "city_reference is missing required columns"
assert not (required_city_meta_cols - set(city_metadata_df.columns)), "city_metadata is missing required columns"

enriched_stream = (
    quality_valid_stream
    .join(city_reference_df.select(*sorted(required_city_ref_cols)), on="city_id", how="left")
    .join(city_metadata_df.select(*sorted(required_city_meta_cols)), on="city_id", how="left")
)
known_city_stream = enriched_stream.filter(col("city_name").isNotNull())
unknown_city_stream = (
    enriched_stream.filter(col("city_name").isNull())
    .withColumn("reject_reason", lit("unknown_city_id"))
)

reject_columns = ["kafka_topic", "partition", "offset", "kafka_timestamp", "kafka_key", "raw_json", "reject_reason"]
rejects_stream = (
    json_rejects_stream.select(*reject_columns)
    .unionByName(quality_rejects_stream.select(*reject_columns))
    .unionByName(unknown_city_stream.select(*reject_columns))
)

print("Explicit event schema:")
event_schema

### Write Bronze, Silver, rejects, and latest snapshot

Finite triggers keep the notebook reproducible. Checkpoints are separate per output. In mock mode, only mock-specific generated outputs and checkpoints are reset before execution.

In [ ]:
def start_finite_parquet_query(stream_df, output_path: Path, checkpoint_path: Path):
    writer = (
        stream_df.writeStream
        .format("parquet")
        .option("path", str(output_path))
        .option("checkpointLocation", str(checkpoint_path))
    )
    try:
        return writer.trigger(availableNow=True).start()
    except TypeError:
        return writer.trigger(once=True).start()

queries = [
    start_finite_parquet_query(json_valid_stream, BRONZE_STREAM_PATH, CHECKPOINT_BRONZE),
    start_finite_parquet_query(rejects_stream, REJECTS_STREAM_PATH, CHECKPOINT_REJECTS),
    start_finite_parquet_query(known_city_stream, SILVER_STREAM_PATH, CHECKPOINT_SILVER),
]
for query in queries:
    query.awaitTermination()
    if query.exception():
        raise RuntimeError(str(query.exception()))

bronze_readback_df = spark.read.parquet(str(BRONZE_STREAM_PATH))
silver_readback_df = spark.read.parquet(str(SILVER_STREAM_PATH))
reject_count = spark.read.parquet(str(REJECTS_STREAM_PATH)).count() if REJECTS_STREAM_PATH.exists() else 0

latest_window = Window.partitionBy("city_id").orderBy(desc("event_time_ts"), desc("ingestion_time_ts"))
latest_snapshot_df = (
    silver_readback_df
    .withColumn("row_number", row_number().over(latest_window))
    .filter(col("row_number") == 1)
    .drop("row_number")
    .withColumn("dataset_context", lit("open_meteo_live"))
)
LATEST_SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
latest_snapshot_df.write.mode("overwrite").parquet(str(LATEST_SNAPSHOT_PATH))
latest_readback_df = spark.read.parquet(str(LATEST_SNAPSHOT_PATH))

result_summary = {
    "selected_source_mode": source_mode,
    "fallback_reason": fallback_reason,
    "bronze_row_count": bronze_readback_df.count(),
    "silver_row_count": silver_readback_df.count(),
    "reject_row_count": reject_count,
    "latest_snapshot_row_count": latest_readback_df.count(),
}
print(result_summary)
silver_readback_df.printSchema()
silver_readback_df.select("city_id", "city_name", "event_time_ts", "data_status", "pm2_5", "pm10", "no2").show(10, truncate=False)

## Validation / Quality Checks

The assertions below validate Spark execution, Parquet read-back, join coverage, deduplication, plausible pollutant ranges, latest-snapshot cardinality, and output location. A mock run must never be reported as the course Kafka evidence.

In [ ]:
silver_count = silver_readback_df.count()
bronze_count = bronze_readback_df.count()
latest_count = latest_readback_df.count()
duplicate_event_ids = silver_readback_df.groupBy("event_id").count().filter(col("count") > 1).count()
unknown_city_count = silver_readback_df.filter(col("city_name").isNull()).count()
invalid_pollutant_count = silver_readback_df.filter(
    ~(col("pm2_5").isNull() | col("pm2_5").between(0, 1000))
    | ~(col("pm10").isNull() | col("pm10").between(0, 2000))
    | ~(col("no2").isNull() | col("no2").between(0, 1000))
).count()

assert bronze_count > 0, "No Bronze rows written. Publish Kafka events or rerun notebook 05 for mock input."
assert silver_count > 0, "No enriched Silver rows written."
assert duplicate_event_ids == 0, f"Duplicate Silver event IDs found: {duplicate_event_ids}"
assert unknown_city_count == 0, f"Unknown city rows leaked into Silver: {unknown_city_count}"
assert invalid_pollutant_count == 0, f"Implausible pollutant rows leaked into Silver: {invalid_pollutant_count}"
assert latest_count <= city_reference_df.select("city_id").distinct().count(), "Latest snapshot contains too many rows"
assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Outputs were written below notebooks/data"

quality_summary_df = silver_readback_df.groupBy("data_status").count().orderBy("data_status")
quality_summary_df.show(truncate=False)
print({
    "spark_read_kafka_requirement_proven": source_mode == "kafka",
    "local_spark_streaming_fallback_tested": source_mode == "mock",
    "silver_rows": silver_count,
    "latest_snapshot_rows": latest_count,
    "reject_rows": reject_count,
})

spark.stop()

## Results

The notebook emits a `result_summary` and a final validation dictionary. For the official FH evidence run, `selected_source_mode` must be `kafka` and `spark_read_kafka_requirement_proven` must be `True`.

For local development, `selected_source_mode=mock` is expected when no broker or Spark Kafka connector is reachable. This still exercises Spark Structured Streaming, explicit JSON parsing, validation, joins, checkpointing, Parquet writes, and read-back.

## Limitations

- A local mock run is not Kafka evidence.
- Remote Spark workers require shared access to `DATA_DIR` and `CHECKPOINT_DIR`; the FH storage layout must be confirmed before remote Parquet claims are made.
- Kafka delivery is at least once; deterministic `event_id` values and Spark deduplication reduce duplicate output.
- Open-Meteo rows marked `controlled_offline_fallback` prove mechanics only and must not support analytical claims.

## Next step
Run notebook `07_gold_layer_and_data_quality.ipynb` to create the analysis-ready Gold layer from historical EEA Silver data and the clearly separated live Open-Meteo snapshot.